In [14]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.linalg as la
from torchvision import datasets, transforms
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm
from torch.utils.data import TensorDataset, DataLoader

# ----------------------------
# Configuration
# ----------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# Problem settings
NUM_CLASSES = 10
FEATURES    = 28 * 28

ValueError: All ufuncs must have type `numpy.ufunc`. Received (<ufunc 'sph_legendre_p'>, <ufunc 'sph_legendre_p'>, <ufunc 'sph_legendre_p'>)

In [2]:
# ----------------------------
# Data
# ----------------------------
def load_mnist():
    # MNIST dataset
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,)),
    ])
    train_dataset = datasets.MNIST(root='./data', train=True, transform=transform, download=True)
    test_dataset = datasets.MNIST(root='./data', train=False, transform=transform, download=True)
    return train_dataset, test_dataset

In [3]:
# ----------------------------
# Model definition
# ----------------------------
class ActNorm(nn.Module):
    """
    Activation Normalization for 4D tensors (images).
    Initializes scale and bias parameters based on the first batch of data.
    """
    def __init__(self, num_channels):
        super().__init__()
        # DO NOT hardcode the device here. Let .to(device) handle it.
        self.scale = nn.Parameter(torch.ones(1, num_channels, 1, 1))
        self.bias = nn.Parameter(torch.zeros(1, num_channels, 1, 1))
        self.register_buffer("initialized", torch.tensor(0, dtype=torch.uint8))

    def forward(self, x):
        # x shape: [B, C, H, W]
        if not self.initialized:
            with torch.no_grad():
                # Calculate mean and stddev over batch and spatial dimensions
                mean = x.mean(dim=[0, 2, 3], keepdim=True)
                std = x.std(dim=[0, 2, 3], keepdim=True)
                self.scale.data.copy_(1.0 / (std + 1e-6))
                self.bias.data.copy_(-mean)
                self.initialized.fill_(1)

        y = self.scale * x + self.bias

        # The log-determinant needs to be scaled by the spatial dimensions
        _, _, h, w = x.size()
        log_det = torch.sum(torch.log(torch.abs(self.scale))) * h * w
        return y, log_det


class Squeeze(nn.Module):
    """
    Squeeze operation that trades spatial dimensions for channel dimensions.
    (B, C, H, W) -> (B, 4*C, H/2, W/2)
    """
    def __init__(self):
        super().__init__()

    def forward(self, x):
        b, c, h, w = x.size()
        x = x.view(b, c, h // 2, 2, w // 2, 2)
        x = x.permute(0, 1, 3, 5, 2, 4).contiguous()
        x = x.view(b, c * 4, h // 2, w // 2)
        return x, 0 # No change to log_det


class CNNCouplingLayer(nn.Module):
    """
    Affine Coupling Layer using a CNN for the coupling function.
    """
    def __init__(self, in_channels, hidden_channels=512):
        super().__init__()
        self.in_channels = in_channels
        self.split_size = in_channels // 2

        # A shallow ResNet-like CNN for the coupling function
        self.coupling_net = nn.Sequential(
            nn.Conv2d(self.split_size, hidden_channels, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(hidden_channels, hidden_channels, kernel_size=1),
            nn.ReLU(),
            # Output 2x channels for s and t
            nn.Conv2d(hidden_channels, self.in_channels, kernel_size=3, padding=1)
        )

        # Initialize last layer to zero for stability
        self.coupling_net[-1].weight.data.zero_()
        self.coupling_net[-1].bias.data.zero_()

    def forward(self, x):
        x_a, x_b = x.split(self.split_size, dim=1)

        s_and_t = self.coupling_net(x_a)
        log_s, t = s_and_t.split(self.split_size, dim=1)
        s = torch.exp(torch.tanh(log_s))

        y_b = x_b * s + t
        y = torch.cat([x_a, y_b], dim=1)

        # Sum log_det over all dimensions except batch
        log_det = torch.sum(torch.log(s), dim=[1, 2, 3])
        return y, log_det


class Invertible1x1Conv(nn.Module):
    """
    Invertible 1x1 Convolution using LU decomposition for initialization.
    Registers an 'initialized' buffer to prevent re-initialization after loading.
    """
    def __init__(self, num_channels):
        super().__init__()
        self.conv = nn.Conv2d(num_channels, num_channels, kernel_size=1, bias=False)

        # Initialize with a random orthogonal matrix for good conditioning
        W = la.qr(torch.randn(num_channels, num_channels))[0]
        self.conv.weight.data.copy_(W.view(num_channels, num_channels, 1, 1))

        # This buffer WILL be saved in the state_dict, solving the problem.
        self.register_buffer("initialized", torch.tensor(1, dtype=torch.uint8))

    def forward(self, x):
        # The weight is already initialized, so we just apply the convolution.
        y = self.conv(x)

        # Calculate log-determinant
        _, _, h, w = x.size()
        log_det = torch.slogdet(self.conv.weight.squeeze())[1] * h * w

        return y, log_det


class DNFNetwork(nn.Module):
    """
    Deep Neural Flow (DNF) Network using CNN Coupling Layers for images.
    """
    def __init__(self, in_channels: int, num_layers: int, hidden_channels: int):
        super(DNFNetwork, self).__init__()

        self.squeeze = Squeeze()
        # After squeeze, channels = 1*4=4, H/W = 28/2=14
        current_channels = in_channels * 4

        self.layers = nn.ModuleList()
        for _ in range(num_layers):
            self.layers.append(ActNorm(current_channels))
            self.layers.append(Invertible1x1Conv(current_channels))
            self.layers.append(CNNCouplingLayer(current_channels, hidden_channels=hidden_channels))

    def forward(self, x):
        # Reshape from flattened to image format
        if len(x.shape) == 2:
            x = x.view(-1, 1, 28, 28)

        total_log_det = torch.zeros(x.shape[0], device=x.device)

        x, log_det = self.squeeze(x)
        total_log_det += log_det

        intermediate_outputs = []

        for layer in self.layers:
            x, log_det = layer(x)
            total_log_det += log_det

            if isinstance(layer, CNNCouplingLayer):
                # We store the flattened z_k and a clone of the current total_log_det
                intermediate_outputs.append(
                    (x.flatten(start_dim=1), total_log_det.clone())
                )

        return intermediate_outputs

In [4]:
# ----------------------------
# Loss definition
# ----------------------------
def compute_logits(z, total_log_det, target_dists):
    log_phi_c = torch.stack([dist.log_prob(z) for dist in target_dists], dim=1)
    logits = log_phi_c + total_log_det.unsqueeze(1)
    return logits

def dnf_loss_dynamic(logits, y_true):
    disc_loss = nn.functional.cross_entropy(logits, y_true)
    return disc_loss

def generative_loss_fn(logits, y_true):
    batch_size = logits.shape[0] # Get batch size dynamically
    true_class_logits = logits[torch.arange(batch_size), y_true]
    gen_loss = -true_class_logits.mean()
    return gen_loss

def deep_supervision_loss(intermediate_logits, y_true, alphas, betas):
    """
    Calculates the total loss with deep supervision.

    Args:
        intermediate_outputs (list): A list of (z_k, log_det_k) tuples from the model.
        y_true (Tensor): The true labels.
        target_dists (list): The list of target distributions.
        alphas (list or Tensor): The weights for the generative loss at each layer.
        betas (list or Tensor): The weights for the auxiliary loss from each layer.

    Returns:
        Tensor: The total computed loss.
    """
    total_loss = torch.tensor(0.0, device=y_true.device)

    for j, logits_j in enumerate(intermediate_logits):
        # Calculate layer-wise hybrid loss
        disc_loss_j = dnf_loss_dynamic(logits_j, y_true)
        gen_loss_j = generative_loss_fn(logits_j, y_true)

        layer_loss = disc_loss_j + alphas[j] * gen_loss_j

        # Add the weighted layer loss to the total
        total_loss += betas[j] * layer_loss

    return total_loss

In [108]:
# ----------------------------
# Training Setup
# ----------------------------
# Model Hyperparameters
NUM_LAYERS = 8              # Number of dense + activation layers
HIDDEN_CHANNELS = 512       # Number of hidden channels in CNN coupling layers

# Dataloader Hyperparameters
BATCH_SIZE = 256

# Initialization Hyperparameters
LATENT_SEPARATION = 2.0

# Training Hyperparameters
EPOCHS = 100
LR_MODEL = 1e-3             # LR for the model
LR_MEANS = 1e-5                # LR for the means
LR_VARS = 1e-5              # LR for the variances
WEIGHT_DECAY = 1e-4         # Using AdamW's weight decay for regularization

# Loss Weights
GAMMA_ALPHA = 0.5
GAMMA_BETA = 0.5
ALPHAS = torch.tensor(np.geomspace(start=GAMMA_ALPHA ** (NUM_LAYERS - 1) * 0.1, stop=0.1, num=NUM_LAYERS), device=device)
BETAS = torch.tensor(np.geomspace(start=GAMMA_BETA ** (NUM_LAYERS - 1), stop=1, num=NUM_LAYERS), device=device)

R_LOGDET = 1e-2             # Regularization strength for the log det
R_VAR = 1e-3                # Regularization strength for the variances

# Instantiate the model and move it to the device
model = DNFNetwork(in_channels=1, num_layers=NUM_LAYERS, hidden_channels=HIDDEN_CHANNELS).to(device)

# Instantiate latent distributions
initial_means = torch.zeros(NUM_CLASSES, FEATURES, device=device)
for i in range(NUM_CLASSES):
    initial_means[i, i] = LATENT_SEPARATION
trainable_means = nn.Parameter(initial_means)

# Define trainable log variances, initialized to zero (i.e., variance of 1).
# We train the log of the variance for numerical stability and to ensure positivity.
initial_log_vars = torch.zeros(NUM_CLASSES, FEATURES, device=device)
trainable_log_vars = nn.Parameter(initial_log_vars)

def get_target_distributions(means_param, log_vars_param):
    # The covariance matrix is now diagonal, with elements exp(log_vars)
    return [
        torch.distributions.MultivariateNormal(
            loc=means_param[i],
            covariance_matrix=torch.diag(torch.exp(log_vars_param[i]))
        ) for i in range(NUM_CLASSES)
    ]

# Use AdamW optimizer which is good for regularization
optimizer = optim.AdamW(model.parameters(), lr=LR_MODEL, weight_decay=WEIGHT_DECAY)
optimizer.add_param_group({'params': [trainable_means], 'lr': LR_MEANS})
optimizer.add_param_group({'params': [trainable_log_vars], 'lr': LR_VARS})
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=25, gamma=0.5)

# Update the learning rate for the original model parameters
for param_group in optimizer.param_groups:
    if 'lr' not in param_group: # The original param group has no explicit LR set yet
        param_group['lr'] = LR_MODEL

# Create TensorDatasets and DataLoaders
train_dataset, test_dataset = load_mnist()
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

In [109]:
# -------------------------------------------------
# Training Loop
# -------------------------------------------------
print("Start training...")

train_losses = []
test_losses = []
test_accuracies = []
test_nlls = []

for epoch in range(EPOCHS):
    model.train()
    total_train_loss = 0.0

    for x_batch, y_batch in train_loader:
        x_batch, y_batch = x_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()

        # Get the current dynamic target distributions using both means and vars
        target_dists = get_target_distributions(trainable_means, trainable_log_vars)

        # Forward pass
        intermediate_outputs = model(x_batch)
        intermediate_logits = [
            compute_logits(z, log_det, target_dists) for z, log_det in intermediate_outputs
        ]
        z, log_det = intermediate_outputs[-1]

        # Combine into a hybrid loss
        loss = deep_supervision_loss(intermediate_logits, y_batch, alphas=ALPHAS, betas=BETAS)

        # Regularization
        # 1. Log-determinant regularization
        loss += R_LOGDET * (log_det ** 2).mean()
        # 2. Variance regularization: Penalize log_vars for deviating from zero
        loss += R_VAR * (trainable_log_vars ** 2).mean()

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        torch.nn.utils.clip_grad_norm_([trainable_means], max_norm=1e-2)
        torch.nn.utils.clip_grad_norm_([trainable_log_vars], max_norm=1e-2)
        optimizer.step()

        total_train_loss += loss.item()

    # --- Evaluation ---
    model.eval()
    total_log_det = 0.0
    total_test_loss = 0.0
    total_nll = 0.0
    correct = 0
    total = 0
    with torch.no_grad():
        # Get dynamic distributions for evaluation
        eval_target_dists = get_target_distributions(trainable_means, trainable_log_vars)

        for x_batch, y_batch in test_loader:
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)
            intermediate_outputs = model(x_batch)
            intermediate_logits = [
                compute_logits(z, log_det, eval_target_dists) for z, log_det in intermediate_outputs
            ]
            z, log_det = intermediate_outputs[-1]
            logits = intermediate_logits[-1]

            # Calculate test loss
            loss = deep_supervision_loss(intermediate_logits, y_batch, alphas=ALPHAS, betas=BETAS)
            loss += R_LOGDET * (log_det ** 2).mean()
            loss += R_VAR * (trainable_log_vars ** 2).mean()
            total_test_loss += loss.item()
            total_log_det += log_det.mean().item()

            # Calculate NLL (a clean evaluation metric)
            nll_batch = nn.functional.cross_entropy(logits, y_batch, reduction='sum')
            total_nll += nll_batch.item()

            # Calculate accuracy
            _, predicted = torch.max(logits.data, 1)

            total += y_batch.size(0)
            correct += (predicted == y_batch).sum().item()

    avg_train_loss = total_train_loss / len(train_loader)
    avg_test_loss = total_test_loss / len(test_loader)
    avg_log_det = total_log_det / len(test_loader)
    avg_nll = total_nll / total

    accuracy = 100 * correct / total

    train_losses.append(avg_train_loss)
    test_losses.append(avg_test_loss)
    test_accuracies.append(accuracy)
    test_nlls.append(avg_nll)

    scheduler.step()

    print(f"Epoch [{epoch+1:02d}/{EPOCHS}] | Train Loss: {avg_train_loss:.4f} | Test Acc: {accuracy:.2f}% | NLL: {avg_nll:.4f} | LogDet: {avg_log_det:.2f}")

print("Finished training.")

Start training...
Epoch [01/100] | Train Loss: 129.7544 | Test Acc: 12.60% | NLL: 2.2742 | LogDet: 7.00
Epoch [02/100] | Train Loss: 84.4602 | Test Acc: 20.00% | NLL: 2.2646 | LogDet: 9.10
Epoch [03/100] | Train Loss: 77.0982 | Test Acc: 24.68% | NLL: 2.2434 | LogDet: 9.00
Epoch [04/100] | Train Loss: 69.9327 | Test Acc: 29.78% | NLL: 2.1834 | LogDet: 5.70
Epoch [05/100] | Train Loss: 57.4461 | Test Acc: 26.29% | NLL: 2.1545 | LogDet: 75.73
Epoch [06/100] | Train Loss: 54.2503 | Test Acc: 32.85% | NLL: 2.0817 | LogDet: -41.51
Epoch [07/100] | Train Loss: 90.7575 | Test Acc: 35.68% | NLL: 2.0566 | LogDet: 10.32
Epoch [08/100] | Train Loss: 75.2079 | Test Acc: 25.41% | NLL: 2.0981 | LogDet: -19.69
Epoch [09/100] | Train Loss: 49.1885 | Test Acc: 37.94% | NLL: 2.0096 | LogDet: 16.74
Epoch [10/100] | Train Loss: 64.1670 | Test Acc: 39.27% | NLL: 1.9806 | LogDet: -53.26
Epoch [11/100] | Train Loss: 56.0844 | Test Acc: 45.49% | NLL: 1.9303 | LogDet: -23.22
Epoch [12/100] | Train Loss: 44.966

In [ ]:
# -------------------------------------------------
# Training Loop
# -------------------------------------------------
print("Start training...")

train_losses = []
test_losses = []
test_accuracies = []
test_nlls = []
trainable_means.requires_grad_(False)
trainable_log_vars.requires_grad_(True)

for epoch in range(EPOCHS):
    model.train()
    total_train_loss = 0.0

    for x_batch, y_batch in train_loader:
        x_batch, y_batch = x_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()

        # Get the current dynamic target distributions using both means and vars
        target_dists = get_target_distributions(trainable_means, trainable_log_vars)

        # Forward pass
        intermediate_outputs = model(x_batch)
        intermediate_logits = [
            compute_logits(z, log_det, target_dists) for z, log_det in intermediate_outputs
        ]
        z, log_det = intermediate_outputs[-1]

        # Combine into a hybrid loss
        loss = deep_supervision_loss(intermediate_logits, y_batch, alphas=ALPHAS, betas=BETAS)

        # Regularization
        # 1. Log-determinant regularization
        loss += R_LOGDET * (log_det ** 2).mean()
        # 2. Variance regularization: Penalize log_vars for deviating from zero
        loss += R_VAR * (trainable_log_vars ** 2).mean()

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        torch.nn.utils.clip_grad_norm_([trainable_means], max_norm=1.0)
        torch.nn.utils.clip_grad_norm_([trainable_log_vars], max_norm=1.0)
        optimizer.step()

        total_train_loss += loss.item()

    # --- Evaluation ---
    model.eval()
    total_log_det = 0.0
    total_test_loss = 0.0
    total_nll = 0.0
    correct = 0
    total = 0
    with torch.no_grad():
        # Get dynamic distributions for evaluation
        eval_target_dists = get_target_distributions(trainable_means, trainable_log_vars)

        for x_batch, y_batch in test_loader:
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)
            intermediate_outputs = model(x_batch)
            intermediate_logits = [
                compute_logits(z, log_det, eval_target_dists) for z, log_det in intermediate_outputs
            ]
            z, log_det = intermediate_outputs[-1]
            logits = intermediate_logits[-1]

            # Calculate test loss
            loss = deep_supervision_loss(intermediate_logits, y_batch, alphas=ALPHAS, betas=BETAS)
            loss += R_LOGDET * (log_det ** 2).mean()
            loss += R_VAR * (trainable_log_vars ** 2).mean()
            total_test_loss += loss.item()
            total_log_det += log_det.mean().item()

            # Calculate NLL (a clean evaluation metric)
            # Use reduction='sum' to accumulate and average later
            nll_batch = nn.functional.cross_entropy(logits, y_batch, reduction='sum')
            total_nll += nll_batch.item()

            # Calculate accuracy
            _, predicted = torch.max(logits.data, 1)

            total += y_batch.size(0)
            correct += (predicted == y_batch).sum().item()

    avg_train_loss = total_train_loss / len(train_loader)
    avg_test_loss = total_test_loss / len(test_loader)
    avg_log_det = total_log_det / len(test_loader)
    avg_nll = total_nll / total

    accuracy = 100 * correct / total

    train_losses.append(avg_train_loss)
    test_losses.append(avg_test_loss)
    test_accuracies.append(accuracy)
    test_nlls.append(avg_nll)

    scheduler.step()

    print(f"Epoch [{epoch+1:02d}/{EPOCHS}] | Train Loss: {avg_train_loss:.4f} | Test Acc: {accuracy:.2f}% | NLL: {avg_nll:.4f} | LogDet: {avg_log_det:.2f}")

print("Finished training.")

Start training...
Epoch [01/100] | Train Loss: -4.5608 | Test Acc: 92.15% | NLL: 0.6800 | LogDet: -8.58
Epoch [02/100] | Train Loss: -3.1633 | Test Acc: 61.22% | NLL: 1.2431 | LogDet: 0.85
Epoch [03/100] | Train Loss: 10.7224 | Test Acc: 91.30% | NLL: 0.7255 | LogDet: -4.55
Epoch [04/100] | Train Loss: -5.8395 | Test Acc: 91.72% | NLL: 0.7064 | LogDet: -22.82
Epoch [05/100] | Train Loss: -5.9514 | Test Acc: 91.82% | NLL: 0.6973 | LogDet: -15.52
Epoch [06/100] | Train Loss: -5.0697 | Test Acc: 91.94% | NLL: 0.6973 | LogDet: 5.82
Epoch [07/100] | Train Loss: -6.6829 | Test Acc: 91.93% | NLL: 0.7004 | LogDet: -19.30
Epoch [08/100] | Train Loss: -6.7057 | Test Acc: 91.84% | NLL: 0.6707 | LogDet: -19.60
Epoch [09/100] | Train Loss: -6.6966 | Test Acc: 91.85% | NLL: 0.7160 | LogDet: -15.85
Epoch [10/100] | Train Loss: -6.8668 | Test Acc: 91.82% | NLL: 0.7071 | LogDet: 9.86
Epoch [11/100] | Train Loss: -2.5209 | Test Acc: 91.87% | NLL: 0.7102 | LogDet: -18.63
Epoch [12/100] | Train Loss: -7.3

In [72]:
# --------------------------------------------
# Save Model Checkpoint
# --------------------------------------------
CHECKPOINT_PATH = "dnf_mnist_checkpoint1.pth"
print(f"Saving checkpoint to {CHECKPOINT_PATH}...")

# Save a dictionary containing the model state, the optimizer state,
# and the final positions of the trainable means and variances.
torch.save({
    'epoch': EPOCHS,
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'trainable_means': trainable_means,
    'trainable_log_vars': trainable_log_vars,
}, CHECKPOINT_PATH)

print("Checkpoint saved successfully.")

Saving checkpoint to dnf_mnist_checkpoint1.pth...
Checkpoint saved successfully.


In [83]:
# ---------------------------------------------------
# Load Checkpoint
# ---------------------------------------------------
BASE_CHECKPOINT_PATH = "dnf_mnist_checkpoint1.pth"

# --- 1. Instantiate a new model and optimizer ---
# This simulates loading the model in a new script or for verification.
model = DNFNetwork(in_channels=1, num_layers=NUM_LAYERS, hidden_channels=HIDDEN_CHANNELS).to(device)
# optimizer = optim.AdamW(model.parameters())

# --- 2. Load the checkpoint file ---
print(f"Loading checkpoint from {BASE_CHECKPOINT_PATH}...")
checkpoint = torch.load(BASE_CHECKPOINT_PATH)

# --- 3. Load the states into the new instances ---
model.load_state_dict(checkpoint['model_state_dict'])
# optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

# Also load the trained means and variances. This is crucial for correct evaluation.
trainable_means = checkpoint['trainable_means'].to(device)
trainable_log_vars = checkpoint['trainable_log_vars'].to(device)

# --- 4. Set the model to evaluation mode ---
model.eval()

print("Model loaded successfully.")

Loading checkpoint from dnf_mnist_checkpoint1.pth...
Model loaded successfully.


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

# Get model predictions for the test set
model.eval()
with torch.no_grad():
    # Recreate the test dataloader to get the full dataset
    _, test_dataset_full = load_mnist()
    X_test_tensor = torch.stack([item[0] for item in test_dataset_full]).to(device)
    y_test = np.array([item[1] for item in test_dataset_full])

    # Define target names for MNIST
    mnist_target_names = [str(i) for i in range(10)]

    # Get the final target distributions
    final_target_dists = get_target_distributions(trainable_means, trainable_log_vars)

    # The model returns a list of intermediate outputs for deep supervision.
    # For final evaluation, we only need the last one.
    intermediate_outputs = model(X_test_tensor)
    z, total_log_det = intermediate_outputs[-1]

    log_phi_c = torch.stack([dist.log_prob(z) for dist in final_target_dists], dim=1)
    logits = log_phi_c + total_log_det.unsqueeze(1)
    _, y_pred = torch.max(logits.data, 1)
    y_pred = y_pred.cpu().numpy()

# Print the classification report
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=mnist_target_names))

# Plot the confusion matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=mnist_target_names, yticklabels=mnist_target_names)
plt.title('Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# ----------------------------
# Calibration Analysis (ECE)
# ----------------------------

# 1. Get model probabilities for the test set
model.eval()
with torch.no_grad():
    # Use the same full test set tensors from the previous cell
    final_target_dists = get_target_distributions(trainable_means, trainable_log_vars)

    # The model returns a list of intermediate outputs. Get the final one for evaluation.
    intermediate_outputs = model(X_test_tensor)
    z, total_log_det = intermediate_outputs[-1]

    log_phi_c = torch.stack([dist.log_prob(z) for dist in final_target_dists], dim=1)
    logits = log_phi_c + total_log_det.unsqueeze(1)

    # Apply softmax to logits to get probabilities
    probabilities = torch.softmax(logits, dim=1)

    # Get the confidence (max probability) and predicted class for each sample
    confidences, predictions = torch.max(probabilities, 1)

# Convert to numpy for calculation
confidences = confidences.cpu().numpy()
predictions = predictions.cpu().numpy()
true_labels = y_test # y_test is already a numpy array from the previous cell

# 2. Calculate ECE
def calculate_ece(confidences, predictions, true_labels, n_bins=15):
    """Calculates the Expected Calibration Error (ECE)."""
    bin_boundaries = np.linspace(0, 1, n_bins + 1)
    bin_lowers = bin_boundaries[:-1]
    bin_uppers = bin_boundaries[1:]

    ece = 0.0
    accuracies_in_bin = []
    avg_conf_in_bin = []
    samples_in_bin_list = []

    for bin_lower, bin_upper in zip(bin_lowers, bin_uppers):
        in_bin = (confidences > bin_lower) & (confidences <= bin_upper)
        prop_in_bin = np.mean(in_bin)
        num_samples = np.sum(in_bin)
        samples_in_bin_list.append(num_samples)

        if prop_in_bin > 0:
            accuracy = np.mean(predictions[in_bin] == true_labels[in_bin])
            avg_confidence = np.mean(confidences[in_bin])
            ece += np.abs(accuracy - avg_confidence) * prop_in_bin

            accuracies_in_bin.append(accuracy)
            avg_conf_in_bin.append(avg_confidence)
        else:
            accuracies_in_bin.append(0)
            avg_conf_in_bin.append(0)

    return ece, accuracies_in_bin, avg_conf_in_bin, samples_in_bin_list, bin_boundaries

ece, accuracies, avg_confs, samples_in_bin, bin_bounds = calculate_ece(confidences, predictions, true_labels)
print(f"Expected Calibration Error (ECE): {ece:.4f}")

# 4. Create and display calibration table
print("\nCalibration Analysis Table:")
cal_data = {
    "Bin": [f"{i+1}" for i in range(len(accuracies))],
    "Confidence Range": [f"{bin_bounds[i]:.2f} - {bin_bounds[i+1]:.2f}" for i in range(len(accuracies))],
    "Avg Confidence": avg_confs,
    "Accuracy": accuracies,
    "Gap": [conf - acc for conf, acc in zip(avg_confs, accuracies)],
    "Samples": samples_in_bin
}
df_cal = pd.DataFrame(cal_data)
df_cal["Avg Confidence"] = df_cal["Avg Confidence"].map('{:.3f}'.format)
df_cal["Accuracy"] = df_cal["Accuracy"].map('{:.3f}'.format)
df_cal["Gap"] = df_cal["Gap"].map('{:+.3f}'.format)

# Display the DataFrame without the index
print(df_cal.to_string(index=False))


# 3. Plot Reliability Diagram
plt.figure(figsize=(8, 8))
plt.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Perfect Calibration')

# Create a bar plot for the accuracies in each bin
n_bins = len(accuracies)
bin_centers = np.linspace(0, 1, n_bins * 2 + 1)[1::2]
bin_width = 1.0 / n_bins

# Plot bars for accuracy
plt.bar(bin_centers, accuracies, width=bin_width, edgecolor='black', alpha=0.6, label='Accuracy')
# Plot bars for confidence (gap)
plt.bar(bin_centers, [c-a for c,a in zip(avg_confs, accuracies)], bottom=accuracies, width=bin_width, edgecolor='black', alpha=0.4, color='red', label='Gap')


plt.title('Reliability Diagram')
plt.xlabel('Confidence')
plt.ylabel('Accuracy')
plt.xlim(0, 1)
plt.ylim(0, 1)
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
from sklearn.metrics import log_loss, brier_score_loss

# --------------------------------------------
# Calculate NLL and Brier Score
# --------------------------------------------

# Ensure the model is in evaluation mode
model.eval()

# Get model probabilities for the test set
# Use the X_test_tensor which contains the entire test dataset
with torch.no_grad():
    # Use the same full test set tensors from the previous cell
    final_target_dists = get_target_distributions(trainable_means, trainable_log_vars)

    # The model returns a list of intermediate outputs. Get the final one for evaluation.
    intermediate_outputs = model(X_test_tensor)
    z, total_log_det = intermediate_outputs[-1]

    log_phi_c = torch.stack([dist.log_prob(z) for dist in final_target_dists], dim=1)
    logits = log_phi_c + total_log_det.unsqueeze(1)

    # Apply softmax to logits to get probabilities
    probabilities = torch.softmax(logits, dim=1).cpu().numpy()

# The true labels are in y_test (numpy array) from the classification report cell

# Calculate Negative Log-Likelihood (NLL)
# log_loss expects true labels (0 to n_classes-1) and predicted probabilities
nll = log_loss(y_test, probabilities)
print(f"\nNegative Log-Likelihood (NLL): {nll:.4f}")

# Calculate Brier Score
# We can calculate the multi-class Brier score as the mean squared difference between
# the predicted probability vector and the one-hot encoded true label vector.

# One-hot encode true labels
y_true_one_hot = np.eye(NUM_CLASSES)[y_test]

# Calculate multi-class Brier Score
brier_score = np.mean(np.sum((probabilities - y_true_one_hot)**2, axis=1))
print(f"Multi-class Brier Score: {brier_score:.4f}")

In [ ]:
import os
import tarfile
import requests
from torchvision.datasets import ImageFolder
from PIL import Image

# --- 1. Function to download and prepare notMNIST ---
def get_notmnist_loader(batch_size=256):
    """Downloads, extracts, and creates a DataLoader for the notMNIST dataset."""
    root = './data'
    url = 'http://yaroslavvb.com/upload/notMNIST/notMNIST_small.tar.gz'
    filename = 'notMNIST_small.tar.gz'
    filepath = os.path.join(root, filename)
    extract_path = os.path.join(root, 'notMNIST_small')

    # Download and extract if it doesn't exist
    if not os.path.exists(extract_path):
        if not os.path.exists(root):
            os.makedirs(root)

        print(f"Downloading {url}...")
        try:
            # Add User-Agent header to mimic a browser request
            headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/58.0.3029.110 Safari/537.3'}
            r = requests.get(url, stream=True, timeout=30, headers=headers)
            r.raise_for_status() # Raise an exception for HTTP errors
            with open(filepath, 'wb') as f:
                for chunk in r.iter_content(chunk_size=8192):
                    f.write(chunk)
            print("Download complete.")

            print(f"Extracting {filepath}...")
            with tarfile.open(filepath, 'r:gz') as tar:
                # Use a filter for security and to prevent warnings in Python 3.14+
                def custom_tar_filter(member, path):
                    if member.name.startswith('/'): # Prevent path traversal
                        return None
                    return member
                tar.extractall(path=root, filter=custom_tar_filter)
            print("Extraction complete.")
            os.remove(filepath)
        except Exception as e:
            print(f"Failed to download or extract notMNIST: {e}")
            return None

    # Define the same transform as MNIST, but ensure images are loaded as grayscale
    transform = transforms.Compose([
        transforms.Grayscale(num_output_channels=1),
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,))
    ])

    # Custom function to check if an image file is valid
    def is_valid_file(path):
        try:
            with Image.open(path) as img:
                img.verify()  # Verify that it is, in fact, an image
            return True
        except (IOError, SyntaxError, Image.UnidentifiedImageError) as e:
            # print(f"Skipping corrupted image: {path} ({e})")
            return False

    try:
        # notMNIST dataset comes with subfolders for each character, which ImageFolder handles automatically
        # Use the custom is_valid_file to filter out corrupted images
        notmnist_dataset = ImageFolder(root=extract_path, transform=transform, is_valid_file=is_valid_file)
        notmnist_loader = DataLoader(notmnist_dataset, batch_size=batch_size, shuffle=False)
        print(f"notMNIST loaded successfully with {len(notmnist_dataset)} valid images.")
        return notmnist_loader
    except Exception as e:
        print(f"Failed to create notMNIST loader: {e}")
        return None

# Create the loader
notmnist_loader = get_notmnist_loader()

In [ ]:
# --- 2. Get model confidences on both MNIST and notMNIST ---

def get_confidences(model, loader, target_dists):
    """Helper function to get model confidences for a given dataset."""
    model.eval()
    all_confidences = []
    with torch.no_grad():
        for x_batch, _ in loader:
            x_batch = x_batch.to(device)

            z, total_log_det = model(x_batch)
            log_phi_c = torch.stack([dist.log_prob(z) for dist in target_dists], dim=1)
            logits = log_phi_c + total_log_det.unsqueeze(1)

            probabilities = torch.softmax(logits, dim=1)
            confidences, _ = torch.max(probabilities, 1)
            all_confidences.append(confidences.cpu())

    return torch.cat(all_confidences).numpy()

# Ensure the fine-tuned model and latent distributions are used
final_target_dists = get_target_distributions(trainable_means, trainable_log_vars)

# Get confidences for the in-distribution MNIST test set
mnist_confidences = get_confidences(model, test_loader, final_target_dists)

# Get confidences for the out-of-distribution notMNIST set
if notmnist_loader:
    notmnist_confidences = get_confidences(model, notmnist_loader, final_target_dists)

    # --- 3. Visualize the confidence distributions ---
    plt.figure(figsize=(12, 6))
    plt.hist(mnist_confidences, bins=50, alpha=0.7, label='In-Distribution (MNIST)', density=True)
    plt.hist(notmnist_confidences, bins=50, alpha=0.7, label='Out-of-Distribution (notMNIST)', density=True)
    plt.title('Model Confidence on In-Distribution vs. Out-of-Distribution Data')
    plt.xlabel('Confidence (Maximum Softmax Probability)')
    plt.ylabel('Density')
    plt.legend()
    plt.grid(True)
    plt.show()

    print(f"Average confidence on MNIST: {np.mean(mnist_confidences):.4f}")
    print(f"Average confidence on notMNIST: {np.mean(notmnist_confidences):.4f}")
else:
    print("Could not run OOD analysis because notMNIST failed to load.")